# 1. Funkce nad poli v NumPy

Nejdřív importujeme `numpy` pod obvyklou zkratkou `np`.


In [1]:
import numpy as np


## 1.1 Základní zpracování dat

NumPy nabízí mnoho funkcí pro práci s daty. Často používané jsou například:
- `np.sort` seřadí pole,
- `np.unique` vrátí unikátní prvky,
- `np.mean` spočítá průměr,
- `np.std` a `np.var` spočítají směrodatnou odchylku a rozptyl,
- `np.sum` a `np.prod` spočítají součet a součin,
- `np.cumsum` a `np.cumprod` vrátí kumulativní součet a součin,
- `np.diff` spočítá rozdíly sousedních prvků.

Přehled je v dokumentaci:
- https://numpy.org/doc/stable/reference/routines.math.html
- https://numpy.org/doc/stable/reference/routines.statistics.html


In [2]:
data = np.random.randint(0, 20, (15))
print(data)


[ 9 14  9 12  8 14 10 11 15  2 19 16 19  0  6]


Setřízené a unikátní prvky pole získáme pomocí `np.sort` a `np.unique`.

In [3]:
print(np.sort(data))
print(np.unique(data))
# pokud bychom chtěli indexy setříděných prvků
print(np.argsort(data))


[ 0  2  6  8  9  9 10 11 12 14 14 15 16 19 19]
[ 0  2  6  8  9 10 11 12 14 15 16 19]
[13  9 14  4  2  0  6  7  3  1  5  8 11 10 12]


### 1.1.1 Průměr (mean)

In [4]:
print(np.mean(data))

10.933333333333334


### 1.1.2 Směrodatná odchylka a rozptyl

In [5]:
print(np.std(data))
print(np.var(data))


5.372357231441541
28.86222222222222


### 1.1.3 Minimum a maximum

In [6]:
print(data.min())
print(data.max())


0
19


### 1.1.4 Součet, součin a stopa

In [7]:
print(np.sum(data))
print(np.prod(data))


164
0


In [8]:
# lze dělat i řádkové a sloupcové součty
A = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(A)
print(np.sum(A, axis=0))
print(np.sum(A, axis=1))


[[1 2 3]
 [4 5 6]
 [7 8 9]]
[12 15 18]
[ 6 15 24]


In [9]:
# stopa matice = součet prvků na diagonále
print(np.trace(A))

15


In [10]:
print(data)

[ 9 14  9 12  8 14 10 11 15  2 19 16 19  0  6]


In [11]:
# kumulativní součet
np.cumsum(data)


array([  9,  23,  32,  44,  52,  66,  76,  87, 102, 104, 123, 139, 158,
       158, 164])

In [12]:
# kumulativní násobení
np.cumprod(data)


array([             9,            126,           1134,          13608,
               108864,        1524096,       15240960,      167650560,
           2514758400,     5029516800,    95560819200,  1528973107200,
       29050489036800,              0,              0])

## 1.2 Iterace

Obecně je lepší preferovat vektorové operace. Někdy je ale potřeba iterovat i explicitně.


In [13]:
v = np.array([1, 2, 3, 4])

for element in v:
    print(element)


1
2
3
4


Iteruje se přes první index (po řádcích).

In [14]:
M = np.array([[1, 2], [3, 4]])

for row in M:
    print(f"row: {row}")


row: [1 2]
row: [3 4]


## 1.3 Vektorizace vlastních funkcí

Vektorizované funkce bývají výrazně rychlejší než explicitní iterace. NumPy nabízí několik cest, jak ze skalární funkce udělat funkci pracující i s polem.


In [15]:
def heaviside_scalar(x):
    """
    Scalar implementation of the Heaviside step function.
    """
    if x >= 0:
        return 1
    return 0

In [16]:
# Toto bychom chtěli, ale pro čistě skalární funkci to nefunguje:
# heaviside_scalar(np.array([-3, -2, -1, 0, 1, 2, 3]))

Jedna možnost je použít `np.vectorize`.

In [17]:
heaviside_vectorized = np.vectorize(heaviside_scalar)

In [18]:
heaviside_vectorized(np.array([-3, -2, -1, 0, 1, 2, 3]))

array([0, 0, 0, 1, 1, 1, 1])

Lepší je funkci přepsat tak, aby přirozeně pracovala se skalárem i polem.

In [19]:
def heaviside_numpy(x):
    """
    Vector-aware implementation of the Heaviside step function.
    """
    return 1.0 * (x >= 0)

In [20]:
heaviside_numpy(np.array([-3, -2, -1, 0, 1, 2, 3]))

array([0., 0., 0., 1., 1., 1., 1.])

In [21]:
# funguje i pro skalár
heaviside_numpy(-1.2), heaviside_numpy(2.6)

(0.0, 1.0)

Porovnejme rychlost obou přístupů.

In [22]:
randvec = np.random.random_sample((10000)) * 2000 - 1000


In [23]:
%timeit heaviside_vectorized(randvec)

940 μs ± 6.11 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [24]:
%timeit heaviside_numpy(randvec)

10.9 μs ± 133 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


## 1.4 Podmínky nad poli

Pokud testujeme podmínku nad celým polem, použijeme typicky `.any()` nebo `.all()`.


In [25]:
M = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])


In [26]:
# výsledkem M > 5 je pole boolovských hodnot
M > 5


array([[False, False, False],
       [False, False,  True],
       [ True,  True,  True]])

In [27]:
if (M > 5).any():
    print("M obsahuje alespoň jeden prvek větší než 5")
else:
    print("M neobsahuje žádný prvek větší než 5")


M obsahuje alespoň jeden prvek větší než 5


In [28]:
if (M > 5).all():
    print("všechny prvky v M jsou větší než 5")
else:
    print("M obsahuje alespoň jeden prvek menší rovno 5")


M obsahuje alespoň jeden prvek menší rovno 5


## 1.5 Sčítání do pole podle indexů

V řadě úloh máme příspěvky a indexy, kam se mají přičíst. Bez smyčky to v NumPy řeší `np.add.at`.


In [29]:
n = 10
m = 1_000_000
idx = np.random.randint(0, n, m)
hodnoty = np.random.rand(m)

In [30]:
y = np.zeros((n,))
for i in range(m):
    y[idx[i]] += hodnoty[i]
    
y

array([50091.39650836, 50141.7211457 , 49954.56114333, 49881.07715716,
       50077.34947045, 49711.62807332, 50031.82144115, 49969.98278592,
       49975.1713234 , 49765.98981469])

In [31]:
x = np.zeros((n,))
np.add.at(x, idx, hodnoty)
x

array([50091.39650836, 50141.7211457 , 49954.56114333, 49881.07715716,
       50077.34947045, 49711.62807332, 50031.82144115, 49969.98278592,
       49975.1713234 , 49765.98981469])

## 1.6 Další čtení

* https://numpy.org/
* https://jakevdp.github.io/PythonDataScienceHandbook/02.00-introduction-to-numpy.html
* http://www.labri.fr/perso/nrougier/teaching/numpy.100/index.html